# Memory-Aware Travel Planning Agent

This notebook demonstrates the **shared implementation** used by Gradio. It needs no API key in demo mode. Travel data are deterministic illustrative mock data. Set `DEMO_MODE=false` and configure an OpenAI key to enable model reasoning with LangChain `create_agent`.

In [ ]:
from pathlib import Path
import sys, uuid
ROOT = Path.cwd() if (Path.cwd() / "app").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from app.config import get_settings
from app.memory_store import MemoryStore
from app.agent import TravelAgent
from app.tools import TRAVEL_TOOLS
from app.planner import create_plan
settings = get_settings()
settings

## 1. Initialize SQLite and shared agent

The database path is resolved relative to the project root unless an absolute path is configured. The user ID scopes preferences; the thread ID scopes conversation history.

In [ ]:
store = MemoryStore(settings.database_path)
agent = TravelAgent(store, settings, force_demo=True)
print("Database:", settings.database_path)
print("Mode:", "DEMO" if agent.demo else "LLM")

## 2. Save and retrieve an explicit profile

Extraction is conservative: only explicit statements become confirmed facts. No budget is filled unless the user provides one.

In [ ]:
demo_user = "notebook-demo-user"
first = agent.respond(demo_user, "first-thread", "I like museums and Italian food, and I avoid early mornings.")
store.get_preferences(demo_user).model_dump()

## 3. Exercise the five shared travel tools

Every result includes a label identifying it as demo data. They are not live travel results.

In [ ]:
tool_inputs = [
    {"origin":"Cairo", "destination":"Florence", "date":"flexible"},
    {"city":"Florence", "budget_max":None},
    {"city":"Florence", "interests":["museums"]},
    {"city":"Florence", "cuisine":"Italian"},
    {"city":"Florence", "date":"upcoming"},
]
[t.invoke(args) for t,args in zip(TRAVEL_TOOLS, tool_inputs)]

## 4. Create the plan before travel tools

The workflow generates a request-specific plan first. Application code controls step order and updates real execution statuses; the model cannot mark tasks complete.

In [ ]:
request = "Plan a weekend itinerary in Florence"
plan = create_plan(request)
[(s.id, s.title, s.status) for s in plan.steps]

## 5. Execute a trip request

The result includes the final response, the validated plan, tool activity records, and the active preference profile.

In [ ]:
result = agent.respond(demo_user, "florence-weekend", request)
print(result["answer"])

### Inspect plan progress and actual tool logs

In [ ]:
[(s.title, s.status, s.error) for s in result["plan"].steps]

In [ ]:
result["tool_logs"]

## 6. Restart persistence demonstration

Save the profile, create a fresh `MemoryStore` instance (which opens SQLite again), and ask from a different thread. No prior thread messages are passed to the new agent.

In [ ]:
# Reopen the persistent database through a fresh store and agent object
store_reopened = MemoryStore(settings.database_path)
agent_reopened = TravelAgent(store_reopened, settings, force_demo=True)
new_thread_result = agent_reopened.respond(demo_user, "new-thread-after-reopen", "Plan a weekend itinerary in Florence")
new_thread_result["preferences"]

In [ ]:
store_reopened.get_messages(demo_user, "new-thread-after-reopen")

The re-opened object retrieves the profile by user ID. The second thread's message list is separate; it does not inherit conversation messages from `first-thread`.

## 7. Configuration and optional LLM mode

For model reasoning and tool selection, set `OPENAI_API_KEY` in `.env` and `DEMO_MODE=false`, then instantiate `TravelAgent(store, settings)`. Keep the key out of notebook output. This notebook pins demo mode for reproducibility and API-free execution.

## 8. Inspect memories and optionally clear this demo user

Deletion is explicit and limited to this demo user. The project never clears persistent data automatically.

In [ ]:
store_reopened.get_preferences(demo_user).model_dump(), store_reopened.get_episodes(demo_user)

In [ ]:
# Optional: uncomment only when you want to erase this notebook demo user's local data
# store_reopened.clear_user(demo_user)

## What this demonstration establishes

- Preferences persist across new store and agent instances because they are in SQLite.
- A new conversation uses its own thread ID and can still retrieve the user's profile.
- Tool outputs and planning statuses are inspectable records.
- Demo mode is explicit and is not represented as live LLM reasoning or live travel data.